# 参考答案：使用 Double DQN 解决 CarRacing-v3


本次作业的目标是在 **Deep Q-Network (DQN)** 的基础上，实现其改进版本 **Double DQN**，训练一个 Agent 来玩 `CarRacing-v3` 游戏。

## 任务说明
由于标准 DQN 处理连续动作较为复杂，本次作业我们将环境配置为 **离散动作空间 (Discrete Action Space)**。

本次作业分为两个部分：

**第一部分（基础）**：完成 DQN 的核心代码填空
1. **CNN 模型构建**：定义用于提取图像特征的卷积神经网络。
2. **动作选择**：实现 $\epsilon$-Greedy (Epsilon-Greedy) 策略。
3. **训练逻辑**：实现 DQN 的核心 Loss 计算公式（Bellman Equation）。

**第二部分（进阶）**：将 DQN 升级为 **Double DQN**
4. **Double DQN 目标值**：用策略网络选择动作、目标网络评估该动作，缓解 Q 值过估计问题。

---

In [1]:
import gymnasium as gym                          # 强化学习环境库（提供 CarRacing-v3）
import matplotlib.pyplot as plt                  # 绘图库
import torch                                     # PyTorch 深度学习框架
import cv2                                       # OpenCV，用于图像处理
import numpy as np                               # 数值计算库
import torch.nn as nn                            # 神经网络模块（Conv2d/Linear/Sequential 等）
import torch.optim as optim                      # 优化器（AdamW 等）
import torch.nn.functional as F                  # 函数式接口（激活函数等）
from collections import namedtuple, deque        # 命名元组与双端队列
from itertools import count                      # 无限计数器
import random                                    # 随机数（探索/采样）
import math                                      # 数学函数（指数衰减）
import os                                        # 系统接口
import imageio                                  # 生成 GIF/视频
from IPython.display import Video               # 在 Notebook 内嵌显示视频

# 自动检测可用设备：有 GPU 用 cuda，否则用 cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. 环境包装器 (Wrapper)

这部分代码已为您提供。它的作用是将原始的 96x96x3 彩色图像转换为 84x84 的灰度图，并将连续 4 帧堆叠在一起作为状态输入。

In [2]:
def image_preprocessing(img):
    img = cv2.resize(img, dsize=(84, 84))                 # 将图像缩放到 84x84
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) / 255.0   # 转为灰度图并归一化到 [0, 1]
    return img

class CarEnvironment(gym.Wrapper):
    def __init__(self, env, skip_frames=3, stack_frames=4, no_operation=50, **kwargs):
        super().__init__(env, **kwargs)                   # 调用父类 Wrapper 的初始化
        self._no_operation = no_operation                 # 重置后"什么都不做"的帧数，等待赛车落地
        self._skip_frames = skip_frames                   # 每个动作重复执行的帧数（帧跳过）
        self._stack_frames = stack_frames                 # 堆叠的帧数（用于捕捉速度/方向信息）
        self.stack_state = None                           # 当前堆叠状态，初始为空

    def reset(self):
        observation, info = self.env.reset()              # 重置底层环境
        # 初始阶段执行 no_operation 步的"空动作"，让赛车在跑道上稳定落地
        for i in range(self._no_operation):
            observation, reward, terminated, truncated, info = self.env.step(0)

        observation = image_preprocessing(observation)    # 对首帧做预处理
        # 初始堆叠：把同一帧重复 stack_frames 次，构成 (4, 84, 84) 状态
        self.stack_state = np.tile(observation, (self._stack_frames, 1, 1))
        return self.stack_state, info

    def step(self, action):
        total_reward = 0
        # 同一个动作在底层环境重复执行 skip_frames 次，累加奖励
        for i in range(self._skip_frames):
            observation, reward, terminated, truncated, info = self.env.step(action)
            total_reward += reward                         # 累加跳帧期间的奖励
            if terminated or truncated:                    # 一旦结束就提前退出
                break

        observation = image_preprocessing(observation)     # 预处理当前帧
        # 滑动窗口：去掉最旧的一帧，拼上最新一帧，保持 4 帧堆叠
        self.stack_state = np.concatenate((self.stack_state[1:], observation[np.newaxis]), axis=0)
        return self.stack_state, total_reward, terminated, truncated, info

## 2. 经验回放 (Replay Memory)

DQN 使用经验回放来打破数据之间的相关性。这部分代码已提供。

In [3]:
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))  # 定义一条转移记录的四元组

class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)          # 固定容量的双端队列，满了自动丢弃最旧样本

    def push(self, *args):
        self.memory.append(Transition(*args))             # 存入一条转移

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)     # 随机采样一个 batch

    def __len__(self):
        return len(self.memory)                           # 返回当前样本数量

## 3. 神经网络构建 (TODO)

你需要定义一个卷积神经网络，输入是状态，输出是每个动作的 Q 值。

* **输入**: `(Batch, 4, 84, 84)`
* **输出**: `(Batch, 5)` (离散动作空间大小为 5)

In [4]:
class CNN(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()                                # 调用 nn.Module 初始化
        # 卷积特征提取层：逐层缩小空间尺寸、增加通道数
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 20, kernel_size=8, stride=4),   # (84-8)/4+1=20 -> (20,20,20)
            nn.ReLU(),
            nn.Conv2d(20, 32, kernel_size=4, stride=2),            # (20-4)/2+1=9  -> (32,9,9)
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1),            # (9-3)/1+1=7   -> (32,7,7)
            nn.ReLU(),
        )

        # 展平后的特征维度：通道数 32 × 高 7 × 宽 7 = 1568
        self._n_features = 32 * 7 * 7

        # 全连接层：把卷积特征映射到每个动作的 Q 值
        self.fc = nn.Sequential(
            nn.Linear(1568,256),                                              # 全连接：1568 -> 256
            nn.ReLU(),                                     # 非线性激活
            nn.Linear(256,out_channels)                           # 输出层：256 -> 动作数(5)
        )

    def forward(self, x):
        x = self.conv(x)                                  # 卷积提取特征
        x = torch.flatten(x, 1)                           # 展平为 (batch, 1568)，保留 batch 维
        x = self.fc(x)                                    # 全连接映射到每个动作的 Q 值
        return x                                          # 返回形状 (batch, n_actions)

## 4. DQN Agent (TODO)

这是作业的核心部分。你需要实现：

**基础部分：**
1. `select_action`: $\epsilon$-Greedy 策略。
2. `train_step`: 从 Memory 中采样，计算 Target Q（标准 DQN），计算 Loss 并更新网络。

**进阶部分（Double DQN）：**
3. 在 `train_step` 的第 4 步中，实现 **Double DQN** 的目标值计算（用策略网络选动作、目标网络估值）。

> 关于 Double DQN 的详细原理，请参考第 5 节的说明。

In [5]:
class DQN_Agent:
    def __init__(self, n_actions, use_double_dqn=True):
        self.n_actions = n_actions                       # 离散动作数量（5 个）
        self.use_double_dqn = use_double_dqn             # 是否使用 Double DQN
        self.batch_size = 256                            # 每次训练的 batch 大小
        self.gamma = 0.99                                # 折扣因子 γ
        self.eps_start = 0.9                             # epsilon 初始值（开始以探索为主）
        self.eps_end = 0.05                              # epsilon 最小值（最终仍有少量探索）
        self.eps_decay = 1000                            # epsilon 指数衰减速率
        self.lr = 1e-4                                   # 学习率

        self.steps_done = 0                              # 已执行步数，用于 epsilon 衰减

        # 策略网络（用于选动作 + 被训练）与目标网络（仅用于计算稳定的目标值）
        self.policy_net = CNN(4, n_actions).to(device)
        self.target_net = CNN(4, n_actions).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())  # 初始时两网络参数完全相同
        self.target_net.eval()                           # 目标网络只做前向推理，不参与训练

        # 优化器只更新策略网络参数
        self.optimizer = optim.AdamW(self.policy_net.parameters(), lr=self.lr, amsgrad=True)
        self.memory = ReplayMemory(10000)                # 经验回放池，容量 10000

    def select_action(self, state, evaluation_phase=False):
        """
                实现 Epsilon-Greedy 策略
                1. 生成一个 0-1 的随机数
                2. 计算当前的 epsilon 阈值 (随时间衰减)
                3. 如果 random > epsilon 或者 evaluation_phase=True -> 选择 Q 值最大的动作 (Exploitation)
                4. 否则 -> 随机选择一个动作 (Exploration)
                """
        sample = random.random()                          # 生成 [0,1) 随机数，决定本次是探索还是利用
        # 计算当前 epsilon 阈值：随步数指数衰减，探索概率逐渐降低
        eps_threshold = self.eps_end + (self.eps_start - self.eps_end) * math.exp(-1. * self.steps_done / self.eps_decay)
        self.steps_done += 1                              # 步数 +1

        # 利用（Exploitation）：选择 Q 值最大的动作
        if evaluation_phase or sample > eps_threshold:
            with torch.no_grad():                         # 推理阶段不计算梯度，节省显存/内存
                return self.policy_net(state).max(1)[1].view(1, 1)  # 取 argmax 动作索引并整形为 (1,1)
        # 探索（Exploration）：随机选择一个动作
        else:
            return torch.tensor([[random.randrange(self.n_actions)]], device=device, dtype=torch.long)

    def train_step(self):
        if len(self.memory) < self.batch_size:            # 经验不足一个 batch 时先不训练
            return

        transitions = self.memory.sample(self.batch_size) # 随机采样一个 batch 的转移
        batch = Transition(*zip(*transitions))            # 把采样结果按字段（state/action/…）重组

        # 非终结状态掩码：next_state 为 None 表示该转移在下一步已结束
        non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), device=device, dtype=torch.bool)
        non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])  # 非终结态的 next_state 堆叠
        state_batch = torch.cat(batch.state)              # 当前状态 batch
        action_batch = torch.cat(batch.action)            # 动作 batch
        reward_batch = torch.cat(batch.reward)            # 奖励 batch

        # ===== 计算当前 Q(s,a) =====
        # 计算当前 Q(s,a)：策略网络输出所有动作的 Q 值，再用 gather 取出实际执行动作对应的 Q 值
        state_action_values = self.policy_net(state_batch).gather(1, action_batch)

        # ===== 计算目标 Q(s,a) =====
        # 初始化目标 Q 值为 0（终结态的目标值保持 0，即 Q_target = r）
        next_state_values = torch.zeros(self.batch_size, device=device)
        with torch.no_grad():                             # 目标值计算不参与梯度反向传播
            if self.use_double_dqn:
                # ===== Double DQN（进阶）=====
                # 步骤：
                #   1) 用策略网络选择动作 a* = argmax_a Q_policy(s', a)
                #   2) 用目标网络评估该动作 Q_target(s', a*)
                # 提示：先用 self.policy_net(...).max(1)[1] 得到动作索引，
                #       再用 self.target_net(...).gather(...) 取对应 Q 值
                # --- YOUR CODE HERE ---
                next_action=self.policy_net(non_final_next_states).max(1)[1].unsqueeze(1)
                next_state_values[non_final_mask] = self.target_net(non_final_next_states).gather(1,next_action).squeeze()
            else:
                # ===== 标准 DQN（基础）=====
                # 公式: Q_target = r + gamma * max_a Q_target(s', a)
                # 提示：直接用 self.target_net(...).max(1)[0] 取最大 Q 值
                # --- YOUR CODE HERE ---
                next_state_values[non_final_next_states]=self.policy_net(non_final_next_states).max(1)[0]

        # Bellman 目标值：Q_target = r + gamma * Q(s', a)
        expected_state_action_values = (next_state_values * self.gamma) + reward_batch

        criterion = nn.SmoothL1Loss()                     # 使用 SmoothL1（Huber）损失，对离群值更鲁棒
        loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))  # 计算损失

        self.optimizer.zero_grad()                        # 清零上一步的梯度
        loss.backward()                                   # 反向传播计算梯度
        torch.nn.utils.clip_grad_value_(self.policy_net.parameters(), 100)  # 梯度裁剪，防止梯度爆炸
        self.optimizer.step()                             # 更新策略网络参数

        return loss.item()                                # 返回当前损失值

    def update_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())  # 把策略网络参数复制到目标网络

## 5. 进阶：Double DQN 原理

### 为什么需要 Double DQN？

标准 DQN 在计算目标值时使用 $\max_a Q_{\text{target}}(s', a)$，即**用同一个目标网络既选择动作、又评估动作**。这会导致 Q 值被系统性地**高估（Overestimation / Maximization Bias）**，进而使训练不稳定、收敛到次优策略。

### Double DQN 的核心思想

Double DQN 将「选择动作」和「评估动作」两个环节**解耦**，交给两个不同的网络：

$$
a^* = \arg\max_a Q_{\text{policy}}(s', a) \qquad \text{(用策略网络选动作)}
$$

$$
Q_{\text{target}} = r + \gamma \cdot Q_{\text{target}}(s', a^*) \qquad \text{(用目标网络估值)}
$$

即：**策略网络**负责选出最优动作 $a^*$，**目标网络**负责评估这个动作的 Q 值，从而有效降低过估计。

### 你的任务

回到上一节 `train_step` 的第 4 步，完成 `if self.use_double_dqn:` 分支的代码，实现 Double DQN 的目标值计算。

## 6. 训练循环

将所有组件组装在一起。请注意我们添加了 **早停机制 (Early Stopping)**，防止车辆在草地上无限打转。

> 提示：`DQN_Agent` 的构造参数 `use_double_dqn` 默认为 `True`（Double DQN）；将其设为 `False` 即可对比标准 DQN 的训练效果。

In [6]:
# 创建离散动作空间的赛车环境，并用 CarEnvironment 包装（帧跳过 + 灰度化 + 4 帧堆叠）
env = gym.make('CarRacing-v3', continuous=False)
env = CarEnvironment(env)

agent = DQN_Agent(n_actions=5)          # 5 个离散动作；use_double_dqn 默认为 True（Double DQN）

episodes = 200                          # 训练总回合数（约 45 分钟）
target_update_frequency = 10            # 每 10 个回合把策略网络同步到目标网络一次

for episode in range(1, episodes + 1):
    state, _ = env.reset()              # 重置环境，获得初始状态
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)  # 转成 tensor 并加 batch 维

    total_reward = 0                    # 累计本回合奖励
    neg_reward_counter = 0              # 连续负奖励计数器（用于早停）

    for t in count():                   # 无限循环，直到回合结束
        action = agent.select_action(state)          # 1. 用 ε-Greedy 选择动作
        next_obs, reward, terminated, truncated, _ = env.step(action.item())  # 2. 执行动作

        # --- 防卡死逻辑：连续负奖励超过 100 步说明车卡在草地，提前终止 ---
        if reward < 0:
            neg_reward_counter += 1     # 负奖励则计数 +1
        else:
            neg_reward_counter = 0      # 正奖励则清零
        if neg_reward_counter > 100:
            truncated = True            # 触发提前终止
            reward = -10.0              # 额外给予较大负奖励作为惩罚

        reward = torch.tensor([reward], device=device)   # 奖励转成 tensor
        done = terminated or truncated                  # 回合是否结束

        if terminated:
            next_state = None           # 终结态：下一步状态记为 None
        else:
            next_state = torch.tensor(next_obs, dtype=torch.float32, device=device).unsqueeze(0)

        agent.memory.push(state, action, next_state, reward)  # 3. 把转移存入经验回放池
        state = next_state              # 更新当前状态
        total_reward += reward.item()   # 累计奖励

        agent.train_step()              # 4. 从回放池采样训练一步

        if done:                        # 回合结束则跳出循环
            print(f"Episode {episode}, Reward: {total_reward:.2f}")
            break

    # 每 target_update_frequency 个回合同步一次目标网络
    if episode % target_update_frequency == 0:
        agent.update_target_network()

Episode 1, Reward: -19.34
Episode 2, Reward: -55.34
Episode 3, Reward: -33.40
Episode 4, Reward: -29.43
Episode 5, Reward: -50.20
Episode 6, Reward: 10.18
Episode 7, Reward: -24.99
Episode 8, Reward: -23.83
Episode 9, Reward: -16.57
Episode 10, Reward: -21.76
Episode 11, Reward: -23.03
Episode 12, Reward: -29.50
Episode 13, Reward: -22.67
Episode 14, Reward: -42.95
Episode 15, Reward: -76.62
Episode 16, Reward: -22.48
Episode 17, Reward: 77.66
Episode 18, Reward: -20.40
Episode 19, Reward: -22.69
Episode 20, Reward: -20.01
Episode 21, Reward: -61.83
Episode 22, Reward: -19.41
Episode 23, Reward: -21.18
Episode 24, Reward: -43.89
Episode 25, Reward: 13.71
Episode 26, Reward: -12.94
Episode 27, Reward: -17.25
Episode 28, Reward: -28.20
Episode 29, Reward: -16.63
Episode 30, Reward: -22.71
Episode 31, Reward: -23.52
Episode 32, Reward: 23.22
Episode 33, Reward: 37.62
Episode 34, Reward: -20.89
Episode 35, Reward: -28.75
Episode 36, Reward: -45.46
Episode 37, Reward: 22.65
Episode 38, Rewa

## 7. 结果可视化

录制视频查看模型效果。

In [7]:
def animate(imgs, video_name):
    if not imgs: return                              # 没有帧则直接返回
    imageio.mimsave(video_name, imgs, fps=30, codec='libvpx-vp9')  # 用 imageio 写 VP9 编码的 webm 视频
    print(f"Video saved as {video_name}")

# 以渲染模式创建评估环境，用于录制模型表现
eval_env = gym.make('CarRacing-v3', continuous=False, render_mode='rgb_array')
eval_env = CarEnvironment(eval_env)
frames = []                        # 存放每一帧图像
obs, _ = eval_env.reset()          # 重置评估环境
done = False

while not done:
    frames.append(eval_env.render())                 # 记录当前帧
    state_tensor = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    action = agent.select_action(state_tensor, evaluation_phase=True)  # 评估模式：纯利用（不探索）
    obs, _, term, trunc, _ = eval_env.step(action.item())  # 执行动作
    done = term or trunc                             # 判断是否结束

eval_env.close()                   # 关闭评估环境
animate(frames, "dqn_result.webm")  # 生成结果视频
Video("dqn_result.webm", width=600, height=400, embed=True)  # 在 Notebook 内嵌播放视频

Video saved as dqn_result.webm


## 8. 思考题（可选加分）

1. Double DQN 相比标准 DQN，为什么能够缓解 Q 值的**过估计 (Overestimation)** 问题？请结合目标值计算公式说明。
2. 将 `use_double_dqn` 设为 `False` 训练一遍，对比两种算法的训练曲线，你观察到了什么差异？

1.DQN公式中Q_target = r + γ · max_a' Q_target(s', a')的“max”会将预测网络中产生的误差一遍遍的扩大，但是Double DQN则将预测与实际值分离开来，通过预测网络和
  目标网络分别求出后再丢给loss，这样一来原DQN中误差放大的问题就得以缓解，即过估计问题得以缓解。

2.标准 DQN 的 Q 值估计偏高且训练曲线更不稳定

